# 02｜TensorFlow：`GradientTape`、回呼與模型保存

對照原筆記的梯度下降、EarlyStopping、ModelCheckpoint 與 SavedModel。使用與第 01 本相同的合成任務；先手寫一步訓練，再展示 `fit` 回呼。

In [ ]:
import tempfile
from pathlib import Path
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)
x = rng.normal(size=(400, 2)).astype('float32')
y = (x[:, 0] + 0.7 * x[:, 1] > 0).astype('int32')
order = rng.permutation(len(x))
x_train, y_train = x[order[:320]], y[order[:320]]
x_valid, y_valid = x[order[320:]], y[order[320:]]
train = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(320, seed=42).batch(32)
valid = tf.data.Dataset.from_tensor_slices((x_valid, y_valid)).batch(32)

In [ ]:
model = tf.keras.Sequential([tf.keras.Input(shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(1e-4)),
    tf.keras.layers.Dropout(0.1), tf.keras.layers.Dense(2)])
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
xb, yb = next(iter(train))
with tf.GradientTape() as tape:
    logits = model(xb, training=True)
    loss = loss_fn(yb, logits) + tf.add_n(model.losses)
grads = tape.gradient(loss, model.trainable_variables)
assert all(g is not None for g in grads)
optimizer.apply_gradients(zip(grads, model.trainable_variables))
print('one training step loss:', float(loss))

`GradientTape` 自動追蹤可訓練權重；驗證時傳 `training=False` 並不更新 optimizer。若使用層上的 L2 regularizer，手寫迴圈需加 `model.losses`。以下改回高階 `fit`，展示回呼。

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
              loss=loss_fn, metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / 'best.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(path, monitor='val_loss', save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ]
    history = model.fit(train, validation_data=valid, epochs=12,
                        callbacks=callbacks, shuffle=False, verbose=0)
    best = tf.keras.models.load_model(path)
    before = best(x_valid[:4], training=False).numpy()
    reloaded = tf.keras.models.load_model(path, compile=False)
    after = reloaded(x_valid[:4], training=False).numpy()
    np.testing.assert_allclose(before, after, rtol=1e-5, atol=1e-5)
    print('epochs:', len(history.history['loss']), 'best val_loss:', min(history.history['val_loss']))
    print('best accuracy:', best.evaluate(valid, verbose=0)[1])

`model.save('model.keras')` 保存可載入的 Keras 模型；`compile=False` 表示載入後沒有編譯，若要繼續 `fit`，要再 `compile`。部署推論則用 `model.export('serving_dir')` 產生 SavedModel；這與 `.keras` 的用途不同。詳見 [TensorFlow 工具書](../../guide/tensorflow.md)。